# 🏁 Motorsport Data Lab — quick start / démarrage rapide

Run every cell (*Runtime → Run all*). Nothing to install on your computer.
Lance toutes les cellules (*Exécution → Tout exécuter*). Rien à installer sur ton ordinateur.

Real timing data: GT3 at the Nürburgring 24h (6 editions) and the French Rallycross Championship.

In [ ]:
# 1. Get the code and the data (about 1 minute)
!git clone --depth 1 https://github.com/DARKQUAD/motorsport-data-lab.git
%cd motorsport-data-lab
%pip install -q ".[models]"

## 2. Nürburgring 24h 2026, GT3: track conditions, pit stops, car models

In [ ]:
import pandas as pd
from motorsport_lab.endurance import laps as lp

df = lp.annotate(pd.read_csv("data/n24/n24_2026_gt3_laps.csv", dtype={"car": str}))
print(f"{df['car'].nunique()} GT3 cars, {len(df)} laps, "
      f"{df['neutralised'].mean():.0%} of laps neutralised or wet")
stops = lp.pit_stops(df)
print(f"median pit-stop loss: {stops['loss_s'].median():.0f} s")
lp.car_model_pace(df).round(1)

## 3. French rallycross 2026: what does the joker lap cost on each track?

In [ ]:
from motorsport_lab.rallycross import analysis as rxa

races = rxa.per_race(pd.read_csv("data/rx_france/rx_france_2026_laps.csv"))
rxa.track_summary(races).round(2)

## 4. Machine learning: next-lap model zoo vs a simple rule (trained h0-16, tested h16-24)
Takes about 30 s. `n_trials=0` skips the Optuna tuning.

In [ ]:
from motorsport_lab.ml import classic

scores, _ = classic.evaluate(df, n_trials=0)
scores.round(2)

## 5. Blind test on other years
Each model only learns from the races *before* the one it is scored on (2017 → 2026).

In [ ]:
from pathlib import Path
from motorsport_lab.ml import seasons

laps = seasons.load_all(Path("data/n24"))
table = seasons.season_table(laps)
errors = seasons.next_lap(table, models=("ridge", "lightgbm", "catboost"))
seasons.summarise(errors, reference=seasons.BASELINES[1]).round(2)

## 6. Who drives the next stint? Learning the drivers' pace during the race
Change the parameters and run again: number of drivers, minimum stints per driver, and how
well last race's driver profile predicts this race (`prior_corr`, measured at 0.07-0.37).

In [ ]:
from motorsport_lab.endurance import lineup

params = lineup.LineupParams(drivers=3, min_stints=4, prior_corr=0.2)
lineup.compare(params, n=500).round(2)

## 7. Optional: Chronos-2 and AutoGluon (heavier, about 10 minutes on a CPU)
```python
%pip install -q ".[ml]"
from motorsport_lab.ml import forecasting as fc
results = fc.backtest(fc.to_series(df), models=("last_clean_lap", "chronos2"))
fc.score(results)
```